# 03｜閱讀時間與測驗：三階提問實作

[![在 Colab 開啟](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/roberthsu2003/PythonForDataAnalysis/blob/master/%E7%9C%8B%E6%87%82%E8%B3%87%E6%96%99%E8%88%87%E9%81%B8%E5%B0%8D%E5%9C%96%E8%A1%A8/%E5%92%8CAI%E5%8D%94%E4%BD%9C%E7%9A%84%E4%B8%89%E9%9A%8E%E6%8F%90%E5%95%8F%E6%B3%95/03_%E9%96%B1%E8%AE%80%E6%99%82%E9%96%93%E8%88%87%E6%B8%AC%E9%A9%97.ipynb)

> **關係 × 散佈圖**｜課堂合成資料；一列代表 **一位匿名學生**。

| ① 看懂資料 | ② 一題一圖 | ③ 三輪追問 |
| :--- | :--- | :--- |
| 欄位與邊界 | 數字表與圖 | 核對數字 → 挑戰推論 → 修訂結論 |

**你負責提問與驗收，AI 負責寫程式。** 先執行第 0 格，依序將 prompt 貼給 Colab Gemini 或能讀取筆記本的 AI；參考程式供核對。第二階完成後，留在同一段對話，接著問第三階的三輪。

**核心問題：**每週閱讀分鐘較多的學生，閱讀測驗分數也較高嗎？　｜　[下載資料集](./資料集/%E9%96%B1%E8%AE%80%E6%99%82%E9%96%93%E8%88%87%E6%B8%AC%E9%A9%97.xlsx)

In [ ]:
# 0｜載入 xlsx，並沿用前面章節的繁體中文字型設定
from pathlib import Path
from urllib.request import urlretrieve
import glob
import os
import pandas as pd
import matplotlib.pyplot as plt

filename = '閱讀時間與測驗.xlsx'
file_path = next((p for p in [Path('資料集') / filename, Path(filename), Path('和AI協作的三階提問法/資料集') / filename] if p.exists()), None)
if file_path is None:
    file_path = Path(filename)
    urlretrieve('https://raw.githubusercontent.com/roberthsu2003/PythonForDataAnalysis/master/%E7%9C%8B%E6%87%82%E8%B3%87%E6%96%99%E8%88%87%E9%81%B8%E5%B0%8D%E5%9C%96%E8%A1%A8/%E5%92%8CAI%E5%8D%94%E4%BD%9C%E7%9A%84%E4%B8%89%E9%9A%8E%E6%8F%90%E5%95%8F%E6%B3%95/%E8%B3%87%E6%96%99%E9%9B%86/%E9%96%B1%E8%AE%80%E6%99%82%E9%96%93%E8%88%87%E6%B8%AC%E9%A9%97.xlsx', file_path)
%pip install -q mplfonts
import mplfonts
from matplotlib import font_manager
from mplfonts import use_font
for font_file in glob.glob(os.path.join(os.path.dirname(mplfonts.__file__), 'fonts', '*.[ot]tf')):
    font_manager.fontManager.addfont(font_file)
use_font('Noto Sans CJK SC')
plt.rcParams['axes.unicode_minus'] = False
df = pd.read_excel(file_path, sheet_name='資料')
print('資料檔：', file_path, '；列數：', len(df))
display(df.head())

---

## ① 第一階｜看懂資料

**先自己判斷：**每週閱讀分鐘、閱讀測驗分數＝指標；班級＝條件；學生代號＝標識。缺分數不能當 0 分。

**複製給 AI**

```text
請讀取已載入的 df（閱讀時間與測驗.xlsx／工作表「資料」）。顯示前 5 列、總列數、欄位名稱、每欄缺值數；說明一列代表什麼，將欄位分成指標、條件、標識。檢查資料的時間範圍、重複或缺值，提出 2 個可以回答、1 個不能回答的問題並說明理由。先不要畫圖，也不要下結論。
```

**驗收：**筆數、缺值與欄位角色，都要和 xlsx 一致。

In [ ]:
print('資料列數：', len(df))
print('欄位：', list(df.columns))
display(df.head(5))
display(df.isna().sum().rename('缺值數').to_frame())

---

## ② 第二階｜一個問題、一張圖

**複製給同一段 AI 對話**

```text
延續第一階，回答「每週閱讀分鐘較多的學生，閱讀測驗分數也較高嗎？」；這是關係問題。先排除每週閱讀分鐘或閱讀測驗分數缺值的列，顯示有效樣本數和兩欄前 5 筆。只畫一張散佈圖（Scatter Plot）：X 軸＝每週閱讀分鐘（分鐘），Y 軸＝閱讀測驗分數（分）。另計算 Pearson 相關係數到小數點後兩位，清楚說它只描述關聯。標題、坐標軸用繁體中文，沿用前面章節的 mplfonts／Noto Sans CJK SC 字型設定。不要畫其他圖。
```

**驗收：**先核對數字表，再核對圖表類型、坐標軸、單位與中文字。

In [ ]:
valid = df.dropna(subset=['每週閱讀分鐘', '閱讀測驗分數']).copy()
display(valid[['學生代號', '每週閱讀分鐘', '閱讀測驗分數']].head())
r = valid['每週閱讀分鐘'].corr(valid['閱讀測驗分數'])
print(f'有效樣本數：{len(valid)}；Pearson r = {r:.2f}')
fig, ax = plt.subplots(figsize=(8, 5.5))
ax.scatter(valid['每週閱讀分鐘'], valid['閱讀測驗分數'], color='#277DA1', alpha=0.8, edgecolor='white', s=70)
ax.set(xlabel='每週閱讀時間（分鐘）', ylabel='閱讀測驗分數（分）', title='閱讀時間與測驗分數的關係')
ax.text(0.03, 0.95, f'有效樣本：{len(valid)} 人｜Pearson r = {r:.2f}', transform=ax.transAxes, va='top')
plt.tight_layout()
plt.show()

---

## ③ 第三階｜接著追問三輪

沿用第二階的數字表與圖，不要重開對話。

### 第 1 輪｜核對數字與排除資料

```text
核對散佈圖點數是否等於兩欄都有值的學生數；列出原始學生數、缺分數數與有效樣本數。確認缺值沒有被補成 0，重新計算並核對 Pearson 相關係數。
```

In [ ]:
missing = df['閱讀測驗分數'].isna().sum()
print('原始學生：', len(df), '；分數缺值：', missing, '；有效樣本：', len(valid), '；Pearson r：', round(r, 2))
assert len(valid) == len(df) - missing
assert valid[['每週閱讀分鐘', '閱讀測驗分數']].notna().all().all()

### 第 2 輪｜挑戰推論

```text
能否說多閱讀會讓分數提高？列出至少兩種可能同時影響閱讀時間和分數、但資料裡沒有的因素。這份樣本能代表所有學生嗎？不要重畫圖。
```

### 第 3 輪｜修訂三句結論

```text
用三句繁體中文寫結論：①根據散佈圖和相關係數描述方向與強弱，附上核對後數值；②有效樣本數與排除的缺值筆數；③相關不等於因果，不能直接推廣到所有學生。避免使用「造成」「證明有效」。
```

> ✍️ **我的驗收紀錄：**哪個數字被核對？哪句推論被改寫？（雙擊此格記下來。）

---

[返回第 7 課](./README.md)　｜　[返回學習地圖](../README.md)